# Basic

In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
import polars as pl
import pickle
import networkx as nx


import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct


In [ ]:
with open(config.ProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)

with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()
Ks = config.TokenizerParam().Ks
rnd_iters = config.TokenizerParam().rnd_iters


# k random

In [ ]:
nodes_df = (pl.DataFrame(
    [{"token": n, **attrs} for n, attrs in combined_subgraphs.nodes(data=True)])
    .with_columns(
        pl.col("token").replace_strict(id_to_label, default=None).alias("label"))
)


In [ ]:
frames = []
for k in Ks:
    for it in config.TokenizerParam().rnd_iters:
        df_sampled = (
            nodes_df.sample(n=k)
            .with_columns(
                pl.lit(k).alias("k"),
                pl.lit(it).alias("iter"))
            .with_row_index()
        )

        frames.append(df_sampled)

df_all = pl.concat(frames)
df_all.write_parquet(config.CandidateLists().k_random_all_samples)

# highest degree and is_a child dataframe construction

In [ ]:
# all relationship distances in the combined subgraph
all_pairs = nx.all_pairs_shortest_path_length(combined_subgraphs)

distance_df = pl.DataFrame(
    [
        {"src_id": src, "dst_id": dst, "distance": dist}
        for src, lengths in all_pairs
        for dst, dist in lengths.items()
        if src != dst
    ]
)
self_distance_df = pl.DataFrame(
    [
        {"src_id": n, "dst_id": n, "distance": 0}
        for n in combined_subgraphs.nodes()
    ]
)

distance_df = pl.concat([distance_df, self_distance_df])

distance_df.write_parquet(config.ProcessedGraph().all_rel_distance_subgraph)


In [ ]:
# is_a only distances in the combined subgraph
is_a_edges = [
    (u, v, k) for u, v, k, attrs in combined_subgraphs.edges(keys=True, data=True)
    if attrs.get("relation") == "IS_A"
]
is_a_subgraph = nx.MultiDiGraph(combined_subgraphs.edge_subgraph(is_a_edges))  # materialize, don't keep it a view

all_pairs = nx.all_pairs_shortest_path_length(is_a_subgraph)
distance_df = pl.DataFrame(
    [
        {"src_id": src, "dst_id": dst, "distance": dist}
        for src, lengths in all_pairs
        for dst, dist in lengths.items()
        if src != dst
    ]
)
self_distance_df = pl.DataFrame(
    [
        {"src_id": n, "dst_id": n, "distance": 0}
        for n in combined_subgraphs.nodes()
    ]
)
distance_df = pl.concat([distance_df, self_distance_df])
distance_df.write_parquet(config.ProcessedGraph().is_a_distance_subgraph)

# highest degree candidate selection

In [ ]:
(pl.read_parquet(config.ProcessedGraph().all_rel_distance_subgraph)
.filter(pl.col("distance") <= config.TokenizerParam().max_dist_candidate)
.group_by("dst_id")
.agg(pl.col("src_id").n_unique().alias("num_in_edges"))
.sort(by = "num_in_edges", descending = True)
.with_row_index()
.rename({"dst_id": "token"})
.write_parquet(config.CandidateLists().highest_degree)
)

In [ ]:
pl.read_parquet(config.ProcessedGraph().all_rel_distance_subgraph)

# highest degree candidate selection dist <= 1

In [ ]:
MAX_DIST = 1
(pl.read_parquet(config.ProcessedGraph().all_rel_distance_subgraph)
.filter(pl.col("distance") <= MAX_DIST)
.group_by("dst_id")
.agg(pl.col("src_id").n_unique().alias("num_in_edges"))
.sort(by = "num_in_edges", descending = True)
.with_row_index()
.rename({"dst_id": "token"})
.write_parquet(config.CandidateLists().highest_degree_dist_1)
)

# most children via is_a

In [ ]:
(pl.read_parquet(config.ProcessedGraph().is_a_distance_subgraph)
.filter(pl.col("distance") <= config.TokenizerParam().max_dist_candidate)
.group_by("dst_id")
.agg(pl.col("src_id").n_unique().alias("num_in_edges"))
.sort(by = "num_in_edges", descending = True)
.with_row_index()
.rename({"dst_id": "token"})
.write_parquet(config.CandidateLists().most_children)

 )